# VQE: the hydrogen molecule

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/vqe-h2.ipynb)

Estimate the ground-state energy of a real molecule. This is the application quantum chemistry is actually chasing.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [VQE: the hydrogen molecule](https://zksf.org/blog/vqe-h2-molecule-ground-state/)


## The idea

The variational quantum eigensolver prepares a trial state on the quantum computer, measures its energy, and lets a classical optimizer adjust the trial until the energy stops falling. The lowest energy it reaches approximates the molecule's ground state.

The molecule here is H2, reduced to a two-qubit Hamiltonian. The energy is not a measurement outcome but an **observable**: a weighted sum of Pauli terms, evaluated on the prepared state.

The angle below is already near optimal, so this notebook shows the final evaluation rather than the whole optimization loop.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

theta = 0.2                  # near-optimal, found by the classical loop

qc = QuantumCircuit(2)       # no measurement: we want an expectation value
qc.x(0)
qc.rx(1.5708, 0)
qc.h(1)
qc.cx(1, 0)
qc.rz(theta, 0)
qc.cx(1, 0)
qc.rx(-1.5708, 0)
qc.h(1)

# The H2 Hamiltonian as weighted Pauli strings.
hamiltonian = [
    [ 0.2333, 'II'], [ 0.3435, 'IZ'], [-0.4347, 'ZI'],
    [ 0.5716, 'ZZ'], [ 0.0910, 'YY'], [ 0.0910, 'XX'],
]
print(qc.draw(output='text'))


## What happened when this ran

This one asks for an expectation value rather than counts, so it ran on the Pauli propagation engine, which evolves the observable instead of the state.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "a5e78c20efb0477a"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

The energy comes out at about **-1.137 hartree**, which is the known ground-state energy of the hydrogen molecule at its equilibrium bond length, the number every quantum chemistry method is checked against.

The error bound is exactly zero: no Pauli terms were discarded during propagation, so the expectation value is exact up to floating point. One honest detail: the identity coefficient folds in the constant nuclear repulsion energy, so this is the full molecular energy you can compare against a textbook, not just the electronic part.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify a5e78c20efb0477a
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000, engine="pauli.cpu")
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000, engine="pauli.cpu",
                 observable=hamiltonian)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/vqe-h2-molecule-ground-state/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
